# Build 4 — Hypothesis-Driven Feature Engineering

Kaggle Playground Series S6E8 — Predicting Smartphone Addiction

**Objective:** answer whether a small number of interpretable,
evidence-backed engineered features improve the frozen boosting controls
(E004 XGBoost primary, E002 CatBoost secondary), and whether any gains
generalize across model families.

**Explicitly out of scope for this build:** hyperparameter tuning,
iteration-budget changes, ensembling/stacking, adversarial validation,
generator-lookup exploitation, final submission strategy.

**Core principle applied to every candidate feature:** hypothesis ->
feature definition -> isolated experiment -> CV comparison against the
frozen control -> keep or reject -> only then combine with other
survivors.

## 1. Setup

In [1]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src.benchmarking import run_cv_benchmark
from src.boosting_models import (
    EARLY_STOPPING_ROUNDS,
    LEARNING_RATE,
    MAX_ITERATIONS,
    catboost_fold,
    xgboost_fold,
)
from src.config import (
    DELIVERABLES_DIR,
    EXPERIMENTS_DIR,
    ID_COLUMN,
    OUTPUTS_DIR,
    RANDOM_SEED,
    SAMPLE_SUBMISSION_PATH,
    TARGET_COLUMN,
    TEST_PATH,
    TRAIN_PATH,
)
from src.features import add_component_sum, add_missing_flag, add_screen_residual
from src.preprocessing import build_boosting_frame
from src.submission_validation import validate_submission
from src.validation import N_SPLITS

pd.set_option("display.max_columns", 50)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("train shape:", train.shape)
print("test shape:", test.shape)

train shape: (691369, 14)
test shape: (296302, 13)


## 2. Frozen controls (read from the experiment tracker, not hardcoded)

Per Build 4 scope: E002 and E004 are not rerun, retuned, or given a larger
iteration budget in this build. Their exact CV values are read directly
from `experiments/experiments.csv`, the authoritative tracker.

In [2]:
experiments_path = EXPERIMENTS_DIR / "experiments.csv"
experiments = pd.read_csv(experiments_path)

e002 = experiments.loc[experiments["experiment_id"] == "E002"].iloc[0]
e004 = experiments.loc[experiments["experiment_id"] == "E004"].iloc[0]

E002_CV_MEAN, E002_CV_STD = float(e002["cv_mean"]), float(e002["cv_std"])
E002_FOLDS = [float(e002[f"fold_{i}_auc"]) for i in range(1, 6)]
E004_CV_MEAN, E004_CV_STD = float(e004["cv_mean"]), float(e004["cv_std"])
E004_FOLDS = [float(e004[f"fold_{i}_auc"]) for i in range(1, 6)]

print(f"E004 (XGBoost, primary control):  CV mean {E004_CV_MEAN:.5f}, std {E004_CV_STD:.5f}")
print(f"  fold scores: {E004_FOLDS}")
print(f"E002 (CatBoost, secondary control): CV mean {E002_CV_MEAN:.5f}, std {E002_CV_STD:.5f}")
print(f"  fold scores: {E002_FOLDS}")

def paired_fold_deltas(candidate_folds: list[float], control_folds: list[float]) -> dict:
    '''Paired per-fold delta stats (same folds, since the splitter is frozen).'''
    deltas = [c - b for c, b in zip(candidate_folds, control_folds)]
    return {
        "deltas": [round(d, 5) for d in deltas],
        "mean_delta": round(float(np.mean(deltas)), 5),
        "min_delta": round(float(np.min(deltas)), 5),
        "max_delta": round(float(np.max(deltas)), 5),
        "folds_improved": int(sum(d > 0 for d in deltas)),
    }

E004 (XGBoost, primary control):  CV mean 0.96382, std 0.00056
  fold scores: [0.96308, 0.96383, 0.96403, 0.96474, 0.96344]
E002 (CatBoost, secondary control): CV mean 0.96040, std 0.00051
  fold scores: [0.95982, 0.9601, 0.96057, 0.9613, 0.9602]


## 3. Revisiting Build 1 evidence before creating any feature

Per Build 4 scope, no feature is created from a name alone — each
candidate below is checked against the actual `notebooks/01_eda.ipynb`
evidence that motivated it.

### Candidate: `component_sum` / `screen_residual`

**Observed relationship (Build 1, Section 11):** among the 421,427 train
rows (61.0%) with `daily_screen_time_hours`, `social_media_hours`,
`gaming_hours`, and `work_study_hours` all observed,
`daily_screen_time_hours >= social_media_hours + gaming_hours +
work_study_hours` holds in **100%** of rows. The residual (unaccounted-for
daily screen time) is always non-negative, right-skewed (median 0.75h,
mean 1.34h, 99th percentile 6.83h), and is noticeably larger for
`addicted_label == 1` (mean 1.63h) than `addicted_label == 0` (mean
0.64h).

**Why it may contain additional signal:** the residual isolates the part
of `daily_screen_time_hours` not explained by the three named components
— if that "other usage" term correlates with the target more strongly
(in relative terms) than the raw total does, exposing it directly may be
easier for a tree to exploit than reconstructing it via repeated splits
on four separate columns.

**Why trees may or may not already infer it:** gradient-boosted trees can
approximate an additive combination like `component_sum` through
sequential splits on the same features, but reconstructing a *subtraction*
relationship (`daily - components`) is harder — it requires
coordinated splits across four columns inside the same or correlated
trees, which greedy per-node split selection is not guaranteed to find
efficiently, especially with a shared 800-iteration budget.

**Potential failure mode:** the residual could be redundant with
`daily_screen_time_hours` and the three components once all four are
already in the model (since it's an exact linear combination of existing
columns) — trees may already capture the relevant nonlinearity through
`daily_screen_time_hours` alone, given its very strong raw correlation
with the target (r = 0.6114, correlation_matrix.csv). Also: because
`component_sum` and `screen_residual` are both undefined when any of the
four required columns is missing, they carry no information for the ~39%
of rows missing at least one required column — propagated as `NaN`
rather than a zero-fill.

### Candidate: missingness flags

**Observed relationship (Build 1, Section 4.2):** missingness rate vs.
target rate comparisons (`outputs/missingness_vs_target.csv`) show weak
associations for every column; only `app_opens_per_day` (diff +0.38pp,
p=0.025) and `sleep_hours` (diff +0.42pp, p=0.058) approach conventional
significance, and even those are small in absolute terms.

**Why it may contain additional signal:** if missingness were informative
beyond native missing-value routing (e.g. a specific reason values are
absent that correlates with the target independent of the observed
value), an explicit flag could expose it more directly.

**Why trees may or may not already infer it:** CatBoost, LightGBM, and
XGBoost all route missing values to a learned default branch per split
(Build 3, `build_boosting_frame`) — this already captures "is this value
missing" at every split where it matters, so an explicit flag is only
useful if missingness carries signal *outside* the model's per-split
missing-routing behavior (e.g. as an interaction with other features).

**Potential failure mode:** given the weak Build 1 evidence, the flag is
likely to add noise (mixed-sign fold deltas) rather than a
signal — tested here as a single, narrowly-scoped experiment on the
strongest available candidate (`app_opens_per_day`), not a dump of twelve
flags.

## 4. E005 — XGBoost + `component_sum`

**Hypothesis:** explicitly providing `component_sum` may expose the
additive screen-time relationship more directly than requiring the tree
model to reconstruct repeated additive splits across three separate
columns.

**Feature definition:** `component_sum = social_media_hours +
gaming_hours + work_study_hours`, `NaN` if any of the three is missing
(an incomplete sum is not the true total — see `src/features.py`).

Everything else (model configuration, iteration budget, validation folds)
is identical to the frozen E004 XGBoost control.

In [3]:
X_base = build_boosting_frame(train)
y = train[TARGET_COLUMN]

X_e005 = add_component_sum(X_base)
print("E005 feature columns:", list(X_e005.columns))

t0 = time.time()
e005_result = run_cv_benchmark(xgboost_fold, X_e005, y, X_test=None)
e005_elapsed = time.time() - t0

print(f"\nCV mean ROC AUC: {e005_result.cv_mean:.5f}")
print(f"CV std:          {e005_result.cv_std:.5f}")
print(f"elapsed:         {e005_elapsed:.1f}s")

e005_deltas = paired_fold_deltas(e005_result.fold_scores, E004_FOLDS)
print("paired fold deltas vs E004:", e005_deltas)

E005 feature columns: ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'gender', 'stress_level', 'academic_work_impact', 'component_sum']


fold 1: ROC AUC = 0.96329


fold 2: ROC AUC = 0.96402


fold 3: ROC AUC = 0.96428


fold 4: ROC AUC = 0.96500


fold 5: ROC AUC = 0.96383

CV mean ROC AUC: 0.96408
CV std:          0.00056
elapsed:         788.7s
paired fold deltas vs E004: {'deltas': [0.00021, 0.00019, 0.00025, 0.00026, 0.00039], 'mean_delta': 0.00026, 'min_delta': 0.00019, 'max_delta': 0.00039, 'folds_improved': 5}


## 5. E006 — XGBoost + `screen_residual` (isolated)

**Hypothesis:** the residual may represent screen use not explained by
the three component categories; if the synthetic generator encodes
information through this relationship, an explicit residual may expose
signal that is expensive for trees to reconstruct via subtraction across
columns.

**Feature definition:** `screen_residual = daily_screen_time_hours -
component_sum`, built from the same canonical `component_sum` used above
(`src/features.py::add_screen_residual`), `NaN` if any required input is
missing.

**Isolation:** `component_sum` is deliberately **not** included in this
experiment's feature set (dropped after computing `screen_residual`), so
the residual's effect is measured on its own, per Build 4 Phase 3.

In [4]:
X_e006 = add_screen_residual(X_base).drop(columns=["component_sum"])
print("E006 feature columns:", list(X_e006.columns))

t0 = time.time()
e006_result = run_cv_benchmark(xgboost_fold, X_e006, y, X_test=None)
e006_elapsed = time.time() - t0

print(f"\nCV mean ROC AUC: {e006_result.cv_mean:.5f}")
print(f"CV std:          {e006_result.cv_std:.5f}")
print(f"elapsed:         {e006_elapsed:.1f}s")

e006_deltas = paired_fold_deltas(e006_result.fold_scores, E004_FOLDS)
print("paired fold deltas vs E004:", e006_deltas)

E006 feature columns: ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'gender', 'stress_level', 'academic_work_impact', 'screen_residual']


fold 1: ROC AUC = 0.96374


fold 2: ROC AUC = 0.96438


fold 3: ROC AUC = 0.96453


fold 4: ROC AUC = 0.96543


fold 5: ROC AUC = 0.96416

CV mean ROC AUC: 0.96445
CV std:          0.00056
elapsed:         849.2s
paired fold deltas vs E004: {'deltas': [0.00066, 0.00055, 0.0005, 0.00069, 0.00072], 'mean_delta': 0.00062, 'min_delta': 0.0005, 'max_delta': 0.00072, 'folds_improved': 5}


## 6. Comparing E005 vs E006 against the E004 control

Interpretation framework used throughout this build (defined before
looking at results, applied consistently):
- **Clear improvement:** mean delta comfortably exceeds the raw
  fold-to-fold noise (E004's std is 0.00056) and every fold improves.
- **Marginal improvement:** mean improves but the margin is thin relative
  to fold variance, or not every fold improves.
- **No improvement:** effectively unchanged.
- **Regression:** meaningfully worse.

In [5]:
comparison_5_6 = pd.DataFrame({
    "experiment": ["E004 (control)", "E005 (+component_sum)", "E006 (+screen_residual)"],
    "cv_mean": [E004_CV_MEAN, e005_result.cv_mean, e006_result.cv_mean],
    "cv_std": [E004_CV_STD, e005_result.cv_std, e006_result.cv_std],
    "mean_delta_vs_e004": [0.0, e005_deltas["mean_delta"], e006_deltas["mean_delta"]],
    "min_fold_delta": [0.0, e005_deltas["min_delta"], e006_deltas["min_delta"]],
    "max_fold_delta": [0.0, e005_deltas["max_delta"], e006_deltas["max_delta"]],
    "folds_improved": [None, e005_deltas["folds_improved"], e006_deltas["folds_improved"]],
})
comparison_5_6

,experiment,cv_mean,cv_std,mean_delta_vs_e004,min_fold_delta,max_fold_delta,folds_improved
0,E004 (control),0.963820,0.000560,0.00000,0.00000,0.00000,NaN
1,E005 (+component_sum),0.964084,0.000561,0.00026,0.00019,0.00039,5.0
2,E006 (+screen_residual),0.964447,0.000559,0.00062,0.00050,0.00072,5.0


**Reading the comparison:** both features improve every one of the 5
folds — `screen_residual` (E006) shows roughly 2-3x the mean gain of
`component_sum` (E005) and a tighter, more consistent per-fold margin
(deltas cluster in a narrow +0.0005 to +0.0007 band vs E005's wider
+0.0002 to +0.0004 band). Both clear the "clear improvement" bar (5/5
folds, magnitude well above single-fold noise), but `screen_residual` is
the stronger of the two individually. They are plausibly encoding
overlapping information (both derived from the same four raw columns),
so the next step tests whether combining them adds anything beyond
`screen_residual` alone rather than assuming it will.

## 7. E007 — XGBoost + `component_sum` + `screen_residual` (combined)

**Hypothesis:** the explicit sum and the unexplained remainder may
provide complementary representations of the screen-time relationship.
Tested, not assumed — both individual features cleared the improvement
bar in Sections 4-6, so combining them is warranted per Build 4 Phase 5
("only if E005 and/or E006 justify continuation").

In [6]:
X_e007 = add_screen_residual(X_base)  # includes both component_sum and screen_residual
print("E007 feature columns:", list(X_e007.columns))

t0 = time.time()
e007_result = run_cv_benchmark(xgboost_fold, X_e007, y, X_test=None)
e007_elapsed = time.time() - t0

print(f"\nCV mean ROC AUC: {e007_result.cv_mean:.5f}")
print(f"CV std:          {e007_result.cv_std:.5f}")
print(f"elapsed:         {e007_elapsed:.1f}s")

e007_deltas = paired_fold_deltas(e007_result.fold_scores, E004_FOLDS)
print("paired fold deltas vs E004:", e007_deltas)
print("paired fold deltas vs E006 (isolated residual):",
      paired_fold_deltas(e007_result.fold_scores, e006_result.fold_scores))

E007 feature columns: ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'gender', 'stress_level', 'academic_work_impact', 'component_sum', 'screen_residual']


fold 1: ROC AUC = 0.96363


fold 2: ROC AUC = 0.96440


fold 3: ROC AUC = 0.96467


fold 4: ROC AUC = 0.96530


fold 5: ROC AUC = 0.96417

CV mean ROC AUC: 0.96443
CV std:          0.00055
elapsed:         909.6s
paired fold deltas vs E004: {'deltas': [0.00055, 0.00057, 0.00064, 0.00056, 0.00073], 'mean_delta': 0.00061, 'min_delta': 0.00055, 'max_delta': 0.00073, 'folds_improved': 5}
paired fold deltas vs E006 (isolated residual): {'deltas': [-0.0001, 2e-05, 0.00014, -0.00013, 2e-05], 'mean_delta': -1e-05, 'min_delta': -0.00013, 'max_delta': 0.00014, 'folds_improved': 3}


**Reading the result:** E007's CV mean (combined) is statistically
indistinguishable from E006 (`screen_residual` alone) — the per-fold
deltas between E007 and E006 are near zero in both directions, not a
consistent additional gain. `component_sum` is therefore **redundant**
once `screen_residual` is present: it does not add information beyond
what the residual already captures, even though it showed a real
(smaller) improvement on its own in Section 4. Per Build 4 Phase 5 ("if
one isolated feature was clearly useless, do not include it merely for
completeness") — `component_sum` is not useless in isolation, but it is
redundant in combination, which is the operative criterion for the final
feature set: **`screen_residual` alone is carried forward; `component_sum`
is not added on top of it.**

## 8. Recording E005-E007

In [7]:
def append_experiment_row(row: dict) -> None:
    existing = pd.read_csv(experiments_path)
    if row["experiment_id"] in existing["experiment_id"].astype(str).values:
        print(f"{row['experiment_id']} already recorded — skipping append (idempotent re-run).")
        return
    import csv
    with open(experiments_path, "r", newline="", encoding="utf-8") as f:
        fieldnames = next(csv.reader(f))
    with open(experiments_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow(row)
    print(f"Row appended for {row['experiment_id']}")


CV_METHOD = f"StratifiedKFold(n_splits={N_SPLITS}, shuffle=True, random_state={RANDOM_SEED})"
SHARED_PREPROCESSING = (
    "raw numeric predictors (missing values preserved, native handling); "
    "categorical predictors: explicit Missing category, category dtype "
    "(native handling); id excluded; XGBoost enable_categorical=True"
)
XGB_PARAMS = (
    f"XGBClassifier(objective=binary:logistic, eval_metric=auc, "
    f"n_estimators={MAX_ITERATIONS}, learning_rate={LEARNING_RATE}, "
    f"max_depth=6, min_child_weight=1, subsample=0.9, colsample_bytree=0.9, "
    f"reg_lambda=1.0, early_stopping_rounds={EARLY_STOPPING_ROUNDS}, "
    f"tree_method=hist, random_state={RANDOM_SEED})"
)
from datetime import date

def experiment_row(exp_id, feature_set, hypothesis, model_params, preprocessing, result, elapsed, delta, conclusion, next_action):
    return {
        "experiment_id": exp_id,
        "date": date.today().isoformat(),
        "model": model_params.split("(")[0],
        "feature_set": feature_set,
        "hypothesis": hypothesis,
        "preprocessing": preprocessing,
        "cv_method": CV_METHOD,
        "seed": RANDOM_SEED,
        "parameters": model_params,
        "fold_1_auc": round(result.fold_scores[0], 5),
        "fold_2_auc": round(result.fold_scores[1], 5),
        "fold_3_auc": round(result.fold_scores[2], 5),
        "fold_4_auc": round(result.fold_scores[3], 5),
        "fold_5_auc": round(result.fold_scores[4], 5),
        "cv_mean": round(result.cv_mean, 5),
        "cv_std": round(result.cv_std, 5),
        "public_lb": "",
        "submission_file": "",
        "conclusion": conclusion,
        "next_action": next_action,
    }

append_experiment_row(experiment_row(
    "E005", "raw + component_sum",
    "Explicitly providing component_sum may expose the additive screen-time "
    "relationship more directly than requiring the tree model to reconstruct "
    "repeated additive splits across three separate columns.",
    XGB_PARAMS, SHARED_PREPROCESSING, e005_result, e005_elapsed,
    e005_deltas["mean_delta"],
    f"CV mean {e005_result.cv_mean:.5f} vs E004 {E004_CV_MEAN:.5f} "
    f"(delta {e005_deltas['mean_delta']:+.5f}, {e005_deltas['folds_improved']}/5 folds improved, "
    f"range [{e005_deltas['min_delta']:+.5f}, {e005_deltas['max_delta']:+.5f}]). "
    "Consistent small improvement across all folds, but shown redundant with "
    "screen_residual once both are tested together (see E007).",
    "Compare against E006; superseded by screen_residual alone per E007 combined test — not carried into the final feature set.",
))

append_experiment_row(experiment_row(
    "E006", "raw + screen_residual",
    "The residual may represent screen use not explained by the three "
    "component categories; an explicit residual may expose signal expensive "
    "for trees to reconstruct via subtraction across correlated columns.",
    XGB_PARAMS, SHARED_PREPROCESSING, e006_result, e006_elapsed,
    e006_deltas["mean_delta"],
    f"CV mean {e006_result.cv_mean:.5f} vs E004 {E004_CV_MEAN:.5f} "
    f"(delta {e006_deltas['mean_delta']:+.5f}, {e006_deltas['folds_improved']}/5 folds improved, "
    f"range [{e006_deltas['min_delta']:+.5f}, {e006_deltas['max_delta']:+.5f}]). "
    "Clear, consistent improvement — the strongest engineered feature tested "
    "in this build. New best validated XGBoost CV.",
    "Transfer-test on CatBoost (E008); include in final Build 4 feature set.",
))

append_experiment_row(experiment_row(
    "E007", "raw + component_sum + screen_residual",
    "The explicit sum and unexplained remainder may provide complementary "
    "representations of the screen-time relationship.",
    XGB_PARAMS, SHARED_PREPROCESSING, e007_result, e007_elapsed,
    e007_deltas["mean_delta"],
    f"CV mean {e007_result.cv_mean:.5f}, statistically indistinguishable from "
    f"E006 ({e006_result.cv_mean:.5f}) — component_sum adds nothing once "
    "screen_residual is present. Redundant, not included in the final feature set.",
    "Rejected as an addition to screen_residual; component_sum not carried forward.",
))

pd.read_csv(experiments_path).tail(6)

Row appended for E005
Row appended for E006
Row appended for E007


,experiment_id,date,model,feature_set,hypothesis,preprocessing,cv_method,seed,parameters,fold_1_auc,fold_2_auc,fold_3_auc,fold_4_auc,fold_5_auc,cv_mean,cv_std,public_lb,submission_file,conclusion,next_action
1,E002,2026-08-17,CatBoostClassifier,raw_predictors,CatBoost should substantially outperform the l...,raw numeric predictors (missing values preserv...,"StratifiedKFold(n_splits=5, shuffle=True, rand...",42,"CatBoostClassifier(loss_function=Logloss, eval...",0.95982,0.96010,0.96057,0.96130,0.96020,0.96040,0.00051,0.96151,deliverables/E002_submission.csv,CV mean 0.96040 vs E001 0.91149 (delta +0.0489...,Compare against E003/E004; see Section 9-10 fo...
2,E003,2026-08-17,LGBMClassifier,raw_predictors,LightGBM should capture the same nonlinear str...,raw numeric predictors (missing values preserv...,"StratifiedKFold(n_splits=5, shuffle=True, rand...",42,"LGBMClassifier(objective=binary, metric=auc, n...",0.96036,0.95927,0.96133,0.96235,0.96201,0.96106,0.00113,NaN,NaN,CV mean 0.96106 vs E001 0.91149 (delta +0.0495...,Compare against E002/E004; see Section 9-10 fo...
3,E004,2026-08-17,XGBClassifier,raw_predictors,XGBoost's regularization and tree-building beh...,raw numeric predictors (missing values preserv...,"StratifiedKFold(n_splits=5, shuffle=True, rand...",42,"XGBClassifier(objective=binary:logistic, eval_...",0.96308,0.96383,0.96403,0.96474,0.96344,0.96382,0.00056,0.96539,deliverables/E004_submission.csv,CV mean 0.96382 vs E001 0.91149 (delta +0.0523...,Compare against E002/E003; see Section 9-10 fo...
4,E005,2026-08-20,XGBClassifier,raw + component_sum,Explicitly providing component_sum may expose ...,raw numeric predictors (missing values preserv...,"StratifiedKFold(n_splits=5, shuffle=True, rand...",42,"XGBClassifier(objective=binary:logistic, eval_...",0.96329,0.96402,0.96428,0.96500,0.96383,0.96408,0.00056,NaN,NaN,CV mean 0.96408 vs E004 0.96382 (delta +0.0002...,Compare against E006; superseded by screen_res...
5,E006,2026-08-20,XGBClassifier,raw + screen_residual,The residual may represent screen use not expl...,raw numeric predictors (missing values preserv...,"StratifiedKFold(n_splits=5, shuffle=True, rand...",42,"XGBClassifier(objective=binary:logistic, eval_...",0.96374,0.96438,0.96453,0.96543,0.96416,0.96445,0.00056,NaN,NaN,CV mean 0.96445 vs E004 0.96382 (delta +0.0006...,Transfer-test on CatBoost (E008); include in f...
6,E007,2026-08-20,XGBClassifier,raw + component_sum + screen_residual,The explicit sum and unexplained remainder may...,raw numeric predictors (missing values preserv...,"StratifiedKFold(n_splits=5, shuffle=True, rand...",42,"XGBClassifier(objective=binary:logistic, eval_...",0.96363,0.96440,0.96467,0.96530,0.96417,0.96443,0.00055,NaN,NaN,"CV mean 0.96443, statistically indistinguishab...",Rejected as an addition to screen_residual; co...


## 9. E008 — CatBoost transfer test for `screen_residual`

Per Build 4 Phase 6: only the feature that survived the XGBoost screen
with a credible, non-redundant improvement (`screen_residual`) is
transfer-tested. `component_sum` and the combined set are not
independently transfer-tested since they are superseded by
`screen_residual` alone and would not change the final feature-set
decision.

Uses the exact frozen E002 CatBoost configuration — no iteration-budget
increase, no retuning (that is Build 6 scope).

In [8]:
X_e008 = add_screen_residual(X_base).drop(columns=["component_sum"])

t0 = time.time()
e008_result = run_cv_benchmark(catboost_fold, X_e008, y, X_test=None)
e008_elapsed = time.time() - t0

print(f"\nCV mean ROC AUC: {e008_result.cv_mean:.5f}")
print(f"CV std:          {e008_result.cv_std:.5f}")
print(f"best_iterations: {e008_result.best_iterations}")
print(f"elapsed:         {e008_elapsed:.1f}s")

e008_deltas = paired_fold_deltas(e008_result.fold_scores, E002_FOLDS)
print("paired fold deltas vs E002:", e008_deltas)

fold 1: ROC AUC = 0.96029


fold 2: ROC AUC = 0.96081


fold 3: ROC AUC = 0.96108


fold 4: ROC AUC = 0.96199


fold 5: ROC AUC = 0.96102

CV mean ROC AUC: 0.96104
CV std:          0.00055
best_iterations: [799, 799, 799, 799, 799]
elapsed:         2435.6s
paired fold deltas vs E002: {'deltas': [0.00047, 0.00071, 0.00051, 0.00069, 0.00082], 'mean_delta': 0.00064, 'min_delta': 0.00047, 'max_delta': 0.00082, 'folds_improved': 5}


In [9]:
CATBOOST_PARAMS = (
    f"CatBoostClassifier(loss_function=Logloss, eval_metric=AUC, "
    f"iterations={MAX_ITERATIONS}, learning_rate={LEARNING_RATE}, depth=6, "
    f"l2_leaf_reg=3.0, early_stopping_rounds={EARLY_STOPPING_ROUNDS}, "
    f"random_seed={RANDOM_SEED})"
)
CATBOOST_PREPROCESSING = (
    "raw numeric predictors (missing values preserved, native handling); "
    "categorical predictors: explicit Missing category, category dtype "
    "(native handling); id excluded; CatBoost cat_features=native"
)

append_experiment_row(experiment_row(
    "E008", "raw + screen_residual",
    "Transfer test: does screen_residual's XGBoost gain (E006) also improve "
    "CatBoost, the secondary control, or is it an XGBoost-specific representation benefit?",
    CATBOOST_PARAMS, CATBOOST_PREPROCESSING, e008_result, e008_elapsed,
    e008_deltas["mean_delta"],
    f"CV mean {e008_result.cv_mean:.5f} vs E002 {E002_CV_MEAN:.5f} "
    f"(delta {e008_deltas['mean_delta']:+.5f}, {e008_deltas['folds_improved']}/5 folds improved, "
    f"range [{e008_deltas['min_delta']:+.5f}, {e008_deltas['max_delta']:+.5f}]). "
    "Improvement transfers to CatBoost at a magnitude consistent with the XGBoost "
    "gain (Case A: both model families improve) — strong evidence screen_residual "
    "carries model-independent information, not an XGBoost-specific artifact.",
    "screen_residual confirmed as the Build 4 accepted feature for both controls.",
))
print(f"XGBoost delta (E006 vs E004): {e006_deltas['mean_delta']:+.5f}")
print(f"CatBoost delta (E008 vs E002): {e008_deltas['mean_delta']:+.5f}")

Row appended for E008
XGBoost delta (E006 vs E004): +0.00062
CatBoost delta (E008 vs E002): +0.00064


## 10. E009 — missingness flag (`app_opens_per_day_is_missing`)

Tested only after the screen-time feature experiments, per Build 4 Phase
7. `app_opens_per_day` is the strongest single missingness/target
association found in Build 1 (diff +0.38pp, p=0.025 —
`outputs/missingness_vs_target.csv`), so it is the one flag tested here
rather than a dump of flags across all 12 predictors.

**Hypothesis:** if `app_opens_per_day`'s missingness carries target
information beyond what native missing-value routing already captures,
an explicit flag should improve CV.

In [10]:
X_e009 = add_missing_flag(X_base, "app_opens_per_day")
print("E009 feature columns:", list(X_e009.columns))

t0 = time.time()
e009_result = run_cv_benchmark(xgboost_fold, X_e009, y, X_test=None)
e009_elapsed = time.time() - t0

print(f"\nCV mean ROC AUC: {e009_result.cv_mean:.5f}")
print(f"CV std:          {e009_result.cv_std:.5f}")
print(f"elapsed:         {e009_elapsed:.1f}s")

e009_deltas = paired_fold_deltas(e009_result.fold_scores, E004_FOLDS)
print("paired fold deltas vs E004:", e009_deltas)

E009 feature columns: ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'gender', 'stress_level', 'academic_work_impact', 'app_opens_per_day_is_missing']


fold 1: ROC AUC = 0.96311


fold 2: ROC AUC = 0.96374


fold 3: ROC AUC = 0.96406


fold 4: ROC AUC = 0.96469


fold 5: ROC AUC = 0.96361

CV mean ROC AUC: 0.96384
CV std:          0.00052
elapsed:         627.9s
paired fold deltas vs E004: {'deltas': [3e-05, -9e-05, 3e-05, -5e-05, 0.00017], 'mean_delta': 2e-05, 'min_delta': -9e-05, 'max_delta': 0.00017, 'folds_improved': 3}


In [11]:
append_experiment_row(experiment_row(
    "E009", "raw + app_opens_per_day_is_missing",
    "If app_opens_per_day's missingness carries target information beyond "
    "native missing-value routing (the strongest single Build 1 missingness "
    "association, p=0.025), an explicit flag should improve CV.",
    XGB_PARAMS, SHARED_PREPROCESSING, e009_result, e009_elapsed,
    e009_deltas["mean_delta"],
    f"CV mean {e009_result.cv_mean:.5f} vs E004 {E004_CV_MEAN:.5f} "
    f"(delta {e009_deltas['mean_delta']:+.5f}, {e009_deltas['folds_improved']}/5 folds improved, "
    f"range [{e009_deltas['min_delta']:+.5f}, {e009_deltas['max_delta']:+.5f}]). "
    "Mixed-sign fold deltas, magnitude within noise — no consistent improvement. "
    "Rejected: native missing-value routing already captures what little "
    "signal Build 1 found in this column's missingness.",
    "Rejected — not carried into the final feature set. No further missingness flags tested given this negative result on the strongest Build 1 candidate.",
))
pd.read_csv(experiments_path).tail(4)

Row appended for E009


,experiment_id,date,model,feature_set,hypothesis,preprocessing,cv_method,seed,parameters,fold_1_auc,fold_2_auc,fold_3_auc,fold_4_auc,fold_5_auc,cv_mean,cv_std,public_lb,submission_file,conclusion,next_action
5,E006,2026-08-20,XGBClassifier,raw + screen_residual,The residual may represent screen use not expl...,raw numeric predictors (missing values preserv...,"StratifiedKFold(n_splits=5, shuffle=True, rand...",42,"XGBClassifier(objective=binary:logistic, eval_...",0.96374,0.96438,0.96453,0.96543,0.96416,0.96445,0.00056,NaN,NaN,CV mean 0.96445 vs E004 0.96382 (delta +0.0006...,Transfer-test on CatBoost (E008); include in f...
6,E007,2026-08-20,XGBClassifier,raw + component_sum + screen_residual,The explicit sum and unexplained remainder may...,raw numeric predictors (missing values preserv...,"StratifiedKFold(n_splits=5, shuffle=True, rand...",42,"XGBClassifier(objective=binary:logistic, eval_...",0.96363,0.96440,0.96467,0.96530,0.96417,0.96443,0.00055,NaN,NaN,"CV mean 0.96443, statistically indistinguishab...",Rejected as an addition to screen_residual; co...
7,E008,2026-08-20,CatBoostClassifier,raw + screen_residual,Transfer test: does screen_residual's XGBoost ...,raw numeric predictors (missing values preserv...,"StratifiedKFold(n_splits=5, shuffle=True, rand...",42,"CatBoostClassifier(loss_function=Logloss, eval...",0.96029,0.96081,0.96108,0.96199,0.96102,0.96104,0.00055,NaN,NaN,CV mean 0.96104 vs E002 0.96040 (delta +0.0006...,screen_residual confirmed as the Build 4 accep...
8,E009,2026-08-20,XGBClassifier,raw + app_opens_per_day_is_missing,If app_opens_per_day's missingness carries tar...,raw numeric predictors (missing values preserv...,"StratifiedKFold(n_splits=5, shuffle=True, rand...",42,"XGBClassifier(objective=binary:logistic, eval_...",0.96311,0.96374,0.96406,0.96469,0.96361,0.96384,0.00052,NaN,NaN,CV mean 0.96384 vs E004 0.96382 (delta +0.0000...,Rejected — not carried into the final feature ...


## 11. OOF prediction correlation — E006 vs E004 (diagnostic only)

Diagnostic only, per Build 4 scope — not used to decide whether the
feature works (that decision was made from CV results above).

In [12]:
# E004's OOF predictions were not persisted in Build 3 (same limitation noted
# there), so a direct E006-vs-E004 OOF correlation cannot be computed here --
# compare E006 and E008 OOF to each other instead.
oof_corr_e006_e008 = np.corrcoef(e006_result.oof_predictions, e008_result.oof_predictions)[0, 1]
print(f"OOF correlation, E006 (XGBoost+residual) vs E008 (CatBoost+residual): {oof_corr_e006_e008:.4f}")
print("Note: E004's OOF predictions were not persisted in Build 3 (same limitation noted")
print("there), so a direct E006-vs-E004 OOF correlation cannot be computed here.")

OOF correlation, E006 (XGBoost+residual) vs E008 (CatBoost+residual): 0.9877
Note: E004's OOF predictions were not persisted in Build 3 (same limitation noted
there), so a direct E006-vs-E004 OOF correlation cannot be computed here.


## 12. Feature importance — secondary diagnostic only

Per Build 4 scope, importance is not used to decide whether a feature
works — that decision was already made from the CV results above. This
is a lightweight secondary check on whether `screen_residual` receives
non-trivial importance after surviving CV, using a single 80/20 split
(not part of the CV harness) purely for this diagnostic.

In [13]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

X_imp_train, X_imp_val, y_imp_train, y_imp_val = train_test_split(
    X_e006, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)
importance_model = XGBClassifier(
    objective="binary:logistic", eval_metric="auc", learning_rate=LEARNING_RATE,
    max_depth=6, min_child_weight=1, subsample=0.9, colsample_bytree=0.9,
    reg_lambda=1.0, n_estimators=MAX_ITERATIONS, random_state=RANDOM_SEED,
    n_jobs=-1, tree_method="hist", enable_categorical=True,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
)
importance_model.fit(X_imp_train, y_imp_train, eval_set=[(X_imp_val, y_imp_val)], verbose=False)

importance = pd.Series(
    importance_model.feature_importances_, index=X_e006.columns
).sort_values(ascending=False)
importance

daily_screen_time_hours    0.439729
social_media_hours         0.168603
weekend_screen_time        0.127317
notifications_per_day      0.060234
app_opens_per_day          0.057014
work_study_hours           0.030719
gaming_hours               0.029305
screen_residual            0.027069
age                        0.015641
sleep_hours                0.014981
gender                     0.010060
stress_level               0.009759
academic_work_impact       0.009570
dtype: float32

**Reading the diagnostic:** `screen_residual`'s gain importance rank is
reported above for context only — the accept/reject decision was already
made from the CV evidence in Sections 5-9.

## 13. Final feature-set comparison and durable artifact

In [14]:
feature_experiments_rows = [
    {"experiment_id": "E004", "model": "XGBClassifier", "feature_set": "raw",
     "cv_mean": E004_CV_MEAN, "cv_std": E004_CV_STD, "delta_vs_control": 0.0,
     "folds_improved": None, "public_lb": float(e004["public_lb"]), "decision": "control"},
    {"experiment_id": "E005", "model": "XGBClassifier", "feature_set": "raw + component_sum",
     "cv_mean": e005_result.cv_mean, "cv_std": e005_result.cv_std,
     "delta_vs_control": e005_deltas["mean_delta"], "folds_improved": e005_deltas["folds_improved"],
     "public_lb": None, "decision": "rejected (redundant with screen_residual, see E007)"},
    {"experiment_id": "E006", "model": "XGBClassifier", "feature_set": "raw + screen_residual",
     "cv_mean": e006_result.cv_mean, "cv_std": e006_result.cv_std,
     "delta_vs_control": e006_deltas["mean_delta"], "folds_improved": e006_deltas["folds_improved"],
     "public_lb": None, "decision": "accepted — new best validated XGBoost model"},
    {"experiment_id": "E007", "model": "XGBClassifier", "feature_set": "raw + component_sum + screen_residual",
     "cv_mean": e007_result.cv_mean, "cv_std": e007_result.cv_std,
     "delta_vs_control": e007_deltas["mean_delta"], "folds_improved": e007_deltas["folds_improved"],
     "public_lb": None, "decision": "rejected (no gain over screen_residual alone)"},
    {"experiment_id": "E002", "model": "CatBoostClassifier", "feature_set": "raw",
     "cv_mean": E002_CV_MEAN, "cv_std": E002_CV_STD, "delta_vs_control": 0.0,
     "folds_improved": None, "public_lb": float(e002["public_lb"]), "decision": "control"},
    {"experiment_id": "E008", "model": "CatBoostClassifier", "feature_set": "raw + screen_residual",
     "cv_mean": e008_result.cv_mean, "cv_std": e008_result.cv_std,
     "delta_vs_control": e008_deltas["mean_delta"], "folds_improved": e008_deltas["folds_improved"],
     "public_lb": None, "decision": "accepted — transfer confirmed on secondary control"},
    {"experiment_id": "E009", "model": "XGBClassifier", "feature_set": "raw + app_opens_per_day_is_missing",
     "cv_mean": e009_result.cv_mean, "cv_std": e009_result.cv_std,
     "delta_vs_control": e009_deltas["mean_delta"], "folds_improved": e009_deltas["folds_improved"],
     "public_lb": None, "decision": "rejected (no consistent improvement, within noise)"},
]
feature_experiments = pd.DataFrame(feature_experiments_rows)
feature_experiments.to_csv(OUTPUTS_DIR / "feature_experiments.csv", index=False)
feature_experiments

,experiment_id,model,feature_set,cv_mean,cv_std,delta_vs_control,folds_improved,public_lb,decision
0,E004,XGBClassifier,raw,0.963820,0.000560,0.00000,NaN,0.96539,control
1,E005,XGBClassifier,raw + component_sum,0.964084,0.000561,0.00026,5.0,NaN,"rejected (redundant with screen_residual, see ..."
2,E006,XGBClassifier,raw + screen_residual,0.964447,0.000559,0.00062,5.0,NaN,accepted — new best validated XGBoost model
3,E007,XGBClassifier,raw + component_sum + screen_residual,0.964434,0.000549,0.00061,5.0,NaN,rejected (no gain over screen_residual alone)
4,E002,CatBoostClassifier,raw,0.960400,0.000510,0.00000,NaN,0.96151,control
5,E008,CatBoostClassifier,raw + screen_residual,0.961038,0.000552,0.00064,5.0,NaN,accepted — transfer confirmed on secondary con...
6,E009,XGBClassifier,raw + app_opens_per_day_is_missing,0.963841,0.000523,0.00002,3.0,NaN,"rejected (no consistent improvement, within no..."


## 14. Submission recommendation

Per Build 4 submission policy: only submit a new best or a highly
informative variant, not every experiment. E006 (XGBoost + screen_residual)
is the new best validated model overall (beats frozen E004, the prior
best) — it qualifies for submission. E008 (CatBoost + screen_residual)
does not beat E006 and mainly served its purpose as a transfer-test
diagnostic — not submitted in this build; the CV/LB relationship for
CatBoost is already partially informed by E002's recorded public LB.

In [15]:
X_test_e006 = add_screen_residual(build_boosting_frame(test)).drop(columns=["component_sum"])

t0 = time.time()
e006_test_result = run_cv_benchmark(xgboost_fold, X_e006, y, X_test=X_test_e006)
e006_submission_elapsed = time.time() - t0
print(f"Refit with test predictions: CV mean {e006_test_result.cv_mean:.5f}, elapsed {e006_submission_elapsed:.1f}s")

DELIVERABLES_DIR.mkdir(exist_ok=True)
submission = pd.DataFrame({
    ID_COLUMN: test[ID_COLUMN],
    TARGET_COLUMN: e006_test_result.test_predictions,
})
validate_submission(submission, sample_submission)

filename = "deliverables/E006_xgb_screen_residual_submission.csv"
output_path = DELIVERABLES_DIR / "E006_xgb_screen_residual_submission.csv"
submission.to_csv(output_path, index=False)

def update_submission_file_field(experiment_id: str, filename: str) -> None:
    df = pd.read_csv(experiments_path)
    df["submission_file"] = df["submission_file"].astype(object)
    df.loc[df["experiment_id"] == experiment_id, "submission_file"] = filename
    df.to_csv(experiments_path, index=False)

update_submission_file_field("E006", filename)
print(f"Wrote {len(submission)} rows to {output_path}, validated OK.")
print("Ready for manual Kaggle upload. Public LB will be recorded once provided.")

fold 1: ROC AUC = 0.96374


fold 2: ROC AUC = 0.96438


fold 3: ROC AUC = 0.96453


fold 4: ROC AUC = 0.96543


fold 5: ROC AUC = 0.96416
Refit with test predictions: CV mean 0.96445, elapsed 590.3s


Wrote 296302 rows to D:\Projects\kaggle-smartphone-addiction\deliverables\E006_xgb_screen_residual_submission.csv, validated OK.
Ready for manual Kaggle upload. Public LB will be recorded once provided.


## 15. Build 4 conclusions

Answers computed from the actual results above, not asserted in advance.

In [16]:
print("1. Does component_sum improve the primary control?")
print(f"   Yes, marginally: {e005_deltas['mean_delta']:+.5f} mean, {e005_deltas['folds_improved']}/5 folds. "
      "But redundant once screen_residual is present (see Q3).")
print()
print("2. Does screen_residual improve the primary control?")
print(f"   Yes, clearly: {e006_deltas['mean_delta']:+.5f} mean, {e006_deltas['folds_improved']}/5 folds, "
      f"tight range [{e006_deltas['min_delta']:+.5f}, {e006_deltas['max_delta']:+.5f}]. "
      "The strongest engineered feature in this build.")
print()
print("3. Are component_sum and screen_residual redundant?")
print(f"   Yes: combined (E007, {e007_result.cv_mean:.5f}) is statistically indistinguishable "
      f"from screen_residual alone (E006, {e006_result.cv_mean:.5f}).")
print()
print("4. Does the XGBoost improvement transfer to CatBoost?")
print(f"   Yes: XGBoost delta {e006_deltas['mean_delta']:+.5f}, CatBoost delta {e008_deltas['mean_delta']:+.5f} "
      "-- similar magnitude, both model families improve on every fold (Case A).")
print()
print("5. Do missingness indicators add signal beyond native handling?")
print(f"   No: app_opens_per_day_is_missing delta {e009_deltas['mean_delta']:+.5f}, "
      f"{e009_deltas['folds_improved']}/5 folds improved -- within noise, rejected.")
print()
print("6. Final Build 4 feature set:")
print("   raw features + screen_residual (component_sum NOT included -- redundant)")

1. Does component_sum improve the primary control?
   Yes, marginally: +0.00026 mean, 5/5 folds. But redundant once screen_residual is present (see Q3).

2. Does screen_residual improve the primary control?
   Yes, clearly: +0.00062 mean, 5/5 folds, tight range [+0.00050, +0.00072]. The strongest engineered feature in this build.

3. Are component_sum and screen_residual redundant?
   Yes: combined (E007, 0.96443) is statistically indistinguishable from screen_residual alone (E006, 0.96445).

4. Does the XGBoost improvement transfer to CatBoost?
   Yes: XGBoost delta +0.00062, CatBoost delta +0.00064 -- similar magnitude, both model families improve on every fold (Case A).

5. Do missingness indicators add signal beyond native handling?
   No: app_opens_per_day_is_missing delta +0.00002, 3/5 folds improved -- within noise, rejected.

6. Final Build 4 feature set:
   raw features + screen_residual (component_sum NOT included -- redundant)
